<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-02-27

| Package | Version |
|---------|---------|
| **nnsight** | **0.6.1** |
| Python | 3.12.11 |
| torch | 2.10.0 |
| transformers | 5.2.0 |

</details>


## Chat Templates

📗 You can find an interactive Colab version of this tutorial [here](https://colab.research.google.com/github/ndif-team/nnsight-website/blob/docs/docs/tutorials/tutorials/get_started/chat_templates.ipynb).

In this tutorial we will be using a [Text Generation Pipeline](https://huggingface.co/docs/transformers/en/conversations#textgenerationpipeline) to demonstrate how to use chat templates with NNsight. We will use Llama 70B-Instruct model remotely to show how different chat formats can be used. With chat templates, rather than writing formatting code by hand each time, you can simply use a template to format chat inputs to any model.


This tutorial was adapted from HuggingFace's [Templates](https://huggingface.co/docs/transformers/en/chat_templating) tutorial.

## Setup

In [1]:
from IPython.display import clear_output
try:
    import google.colab
    is_colab = True
    !pip install nnsight
    !pip install msgspec python-socketio[client]
except ImportError:
    is_colab = False

clear_output()

In [2]:
import nnsight
from nnsight import CONFIG
nnsight.CONFIG.APP.REMOTE_LOGGING = False
from nnsight import LanguageModel, util
import os
import torch
from transformers import AutoTokenizer

if is_colab:
    # include your HuggingFace Token and NNsight API key on Colab secrets
    from google.colab import userdata
    NDIF_API_KEY = userdata.get('NDIF_API')
    CONFIG.set_default_api_key(NDIF_API_KEY)
clear_output()

In [3]:
# load in LLama-70B Instruct model
model = LanguageModel("meta-llama/Llama-3.3-70B-Instruct", device_map="auto", dtype="bfloat16")

## Applying a Chat Template

In order to apply a chat template, there is a specific format that each conversation piece should take. They should be formatted as a list of dictionaries with `role` and `content` key value pairs.

![CHAT TEMPLATE EXAMPLE](https://github.com/ndif-team/nnsight-website/blob/docs/docs/tutorials/tutorials/images/chat-template-ex.png?raw=1)

The `role` key is used to specify the speaker and the `content` key is used to describe how the model should respond when prompted by the given speaker. In order to apply the template we need to load the tokenizer from the LLama 70B Instruct model so that the model can convert tokens into text. The `user` role is the human asking the question and the `assistant` role provides context to the model about what has already been "said".



Now we will pass a messager to our model and see how it responds!

In [4]:
# load in tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.3-70B-Instruct")

# define chat conversation
chat = [
    {"role": "system", "content": "You are a friendly chatbot who always responds like a teacher"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]

# convert the conversation into a format the model will understand
prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True, return_tensors='pt')

print(tokenizer.decode(prompt['input_ids'][0]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a friendly chatbot who always responds like a teacher<|eot_id|><|start_header_id|>user<|end_header_id|>

How many helicopters can a human eat in one sitting?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




To generate a response, we will use `.generate()` and `nnsight`'s remote execution

In [5]:
with model.generate(prompt, max_new_tokens=128, remote=True) as gen:
    # save final generated tokens
    saved = model.generator.output.save()

# print each decoded output on a new line
for seq in saved:
    print(model.tokenizer.decode(seq, skip_special_tokens=True))

⬇ Downloading:   0%|          | 0.00/1.05k [00:00<?]

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a friendly chatbot who always responds like a teacheruser

How many helicopters can a human eat in one sitting?assistant

My inquisitive student, I must say that's a rather... interesting question! However, I must clarify that it's not possible for a human to eat a helicopter, as they are large machines made of metal, plastic, and other materials, not edible substances.

In fact, attempting to consume a helicopter would be extremely hazardous to one's health, not to mention physically impossible. Helicopters are complex vehicles designed for transportation, not for human consumption.

So, to answer your question in a polite and educational manner, the correct response would be "zero." A human cannot eat a helicopter in one sitting, or in any sitting,


As you can see, the prompt we give the chat model has a huge influence on how the model responds. Let's try some other prompts to see the difference in what the model generates:

In [6]:
# change the system prompt s
chat = [
    {"role": "system", "content": "You are a serious chatbot who always responds like a news reporter"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]

prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True, return_tensors='pt')

with model.generate(prompt, max_new_tokens=128, remote=True) as gen:
    saved = model.generator.output.save()

for seq in saved:
    print(model.tokenizer.decode(seq, skip_special_tokens=True))

⬇ Downloading:   0%|          | 0.00/1.07k [00:00<?]

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a serious chatbot who always responds like a news reporteruser

How many helicopters can a human eat in one sitting?assistant

(Breaking News Theme Music Plays)

Good evening, I'm reporting live from our newsroom, where we've received a rather...unconventional question. We're going to tackle this inquiry with the seriousness and professionalism that our viewers have come to expect from our network.

After conducting a thorough investigation, our team of experts has concluded that it is not possible for a human to consume a helicopter in one sitting, or indeed, at all. Helicopters are complex machines made of metal, plastic, and other materials, and are not edible or digestible by humans.

In fact, attempting to ingest a helicopter would be extremely hazardous and


## Chat Template Parameters



### Indicating the Start of a Response


If you'd like to indicate the start of a response, you can use the `add_generation_prompt`. This prompt ensures the model will generate a system response rather than continue the user's message.

In [7]:
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False, return_tensors='pt')
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a serious chatbot who always responds like a news reporter<|eot_id|><|start_header_id|>user<|end_header_id|>

How many helicopters can a human eat in one sitting?<|eot_id|>


*Note: Not all models require generation prompts*



### Continuing the Final Messages

In a similar way, the `continue_final_message` parameter determines whether the message should be continued or whether a new message should start. This parameter is useful if you want to 'prefill' a model response to improve the accuracy of a particular instruction.  

In [8]:
final_chat = [
    {"role": "user", "content": "Can you format the answer in JSON?"},
    {"role": "assistant", "content": '{"name": "'},
]

formatted_chat = tokenizer.apply_chat_template(final_chat, tokenize=True, return_dict=True, continue_final_message=True, return_tensors='pt')

with model.generate(formatted_chat, remote = True):
    output = model.generator.output.save()

for seq in output:
    print(model.tokenizer.decode(seq, skip_special_tokens=True))


⬇ Downloading:   0%|          | 0.00/785 [00:00<?]

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Can you format the answer in JSON?assistant

{"name": "example", "message": "I'd be happy to help you with your question. However, you


*Note: You shouldn’t use `add_generation_prompt` and `continue_final_message` at the same time. The `add_generation_prompt` adds tokens that start a new message, while the latter removes end of sequence tokens. Using them together returns an error.*

## Multiple Templates

Each model may have several different templates available for use. In the case that there are multiple templates, the chat template can serve as a dictionary where each dictionary key corresponds to a specific template name. However, there will always be a default template which the `apply_chat_template` command will always look for.


In order to access other templates, you can simply add the `chat_template` parameter and the name of whichever template you wish to use.

## Model Training

Chat templates can be added as a preprocessing step before model training as a way to ensure a chat template matches the tokens a model is trained on. You can set `add_generation_prompt= False` because extra tokens meant to start a reply aren't needed while training.
<br>

It is important to note that some tokenizers add special `<bos>` and `<eos>` tokens. However, adding additional special tokens is often incorrect or duplicated, hurting model performance. When you format text with `apply_chat_template(tokenize = False)`, make sure you set `add_special_tokens = False` as well to avoid duplicating them.

